In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Define a transform to normalize the data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Download and load the full training data
full_trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Download and load the full test data
testset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)

# Define the function to split training data
def split_train_data(full_trainset, num_subsets=10, split_by_digit=False):
    if split_by_digit:
        digit_indices = [[] for _ in range(10)]
        for i, (_, label) in enumerate(full_trainset):
            digit_indices[label].append(i)
        
        subsets = []
        for i in range(10):
            # Ensure that we only create subsets for digits that have samples
            if digit_indices[i]: # only create subset if there are samples for this digit
                subsets.append(torch.utils.data.Subset(full_trainset, digit_indices[i]))

    else:
        num_total_samples = len(full_trainset)
        indices = list(range(num_total_samples))
        np.random.shuffle(indices)
        
        subsets = []
        samples_per_subset = num_total_samples // num_subsets
        for i in range(num_subsets):
            start_idx = i * samples_per_subset
            # For the last subset, include any remaining samples
            end_idx = (i + 1) * samples_per_subset if i < num_subsets - 1 else num_total_samples
            subsets.append(torch.utils.data.Subset(full_trainset, indices[start_idx:end_idx]))
            
    return subsets

In [ ]:
# Define the CNN architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # First convolutional layer (1 input channel, 32 output channels, 3x3 kernel, stride 1, padding 1)
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        # Max pooling layer (2x2 kernel, stride 2)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional layer (32 input channels, 64 output channels, 3x3 kernel, stride 1, padding 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        # Max pooling layer (2x2 kernel, stride 2)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        # MNIST images are 28x28. After two 2x2 pooling layers, size becomes 28/2 -> 14/2 -> 7x7
        # So, input features to the linear layer = 64 channels * 7 * 7 image size
        self.fc1 = nn.Linear(64 * 7 * 7, 128) 
        self.fc2 = nn.Linear(128, 10) # Output layer (10 classes for MNIST digits)

    def forward(self, x):
        # Apply first conv layer, then ReLU, then pooling
        x = self.pool1(F.relu(self.conv1(x)))
        # Apply second conv layer, then ReLU, then pooling
        x = self.pool2(F.relu(self.conv2(x)))
        
        # Flatten the output from conv layers before passing to linear layers
        x = x.view(-1, 64 * 7 * 7) # -1 infers batch size
        
        # Apply first linear layer, then ReLU
        x = F.relu(self.fc1(x))
        # Apply second linear layer (output layer)
        x = self.fc2(x)
        return x

# Instantiate the model
model = Net()

# Print model structure (optional)
print(model)

In [ ]:
# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Configuration for data splitting
SPLIT_BY_DIGIT = False # Set to True to split by digit, False for random subsets
NUM_SUBSETS = 10
SUBSET_NUM_EPOCHS = 3 # Number of epochs to train on each subset (reduced for faster iteration)

# Prepare the subsets
# Ensure full_trainset is available from the 'load-preprocess-data' cell
# Ensure split_train_data function is available from the 'load-preprocess-data' cell
train_subsets = split_train_data(full_trainset, num_subsets=NUM_SUBSETS, split_by_digit=SPLIT_BY_DIGIT)

overall_val_accuracies_after_subset_training = []
# model, criterion, optimizer should be defined in 'define-model' and 'compile-model' cells

print(f"Starting training process on {NUM_SUBSETS} subsets...")
if SPLIT_BY_DIGIT:
    print("Data is split by digit.")
else:
    print("Data is split into random, non-overlapping subsets.")

for subset_idx, current_subset_data in enumerate(train_subsets):
    print(f"--- Training on Subset {subset_idx + 1}/{NUM_SUBSETS} ---")
    
    if len(current_subset_data) == 0:
        print(f"Subset {subset_idx + 1} is empty. Skipping.")
        # Append a placeholder for accuracy if needed, e.g., 0 or None, or handle in plotting
        overall_val_accuracies_after_subset_training.append(0) # Or None
        continue

    current_trainloader = torch.utils.data.DataLoader(current_subset_data, batch_size=64, shuffle=True)
    
    subset_train_losses = [] # To store losses for epochs within this subset

    for epoch in range(SUBSET_NUM_EPOCHS):
        model.train() # Set model to training mode
        running_loss = 0.0
        for i, data in enumerate(current_trainloader, 0):
            images, labels = data
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        epoch_loss = running_loss / len(current_trainloader)
        subset_train_losses.append(epoch_loss)
        print(f'Subset {subset_idx + 1}, Epoch [{epoch + 1}/{SUBSET_NUM_EPOCHS}], Training Loss: {epoch_loss:.4f}')

    # Report training loss for the subset (last epoch's loss)
    last_epoch_loss = subset_train_losses[-1] if subset_train_losses else float('nan')
    print(f'--- Finished training on Subset {subset_idx + 1} ---')
    print(f'Last Training Loss for Subset {subset_idx + 1}: {last_epoch_loss:.4f}')
    
    # Validation on the full test set
    model.eval() # Set model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader: # testloader uses the full test set
            images, labels = data
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    current_val_accuracy = 100 * correct / total
    overall_val_accuracies_after_subset_training.append(current_val_accuracy)
    print(f'Validation Accuracy on Full Test Set after Subset {subset_idx + 1}: {current_val_accuracy:.2f} %')
    
    # Display a few samples from the current training subset
    print(f'Displaying samples from Subset {subset_idx + 1}:')
    model.eval() # Keep model in eval mode for consistency
    num_samples_to_show = 3
    
    # Ensure current_subset_data is not empty before trying to get samples
    if len(current_subset_data) > 0:
        # Get a few random samples from the current_subset_data
        sample_indices = np.random.choice(len(current_subset_data), min(num_samples_to_show, len(current_subset_data)), replace=False)
        
        fig, axes = plt.subplots(1, len(sample_indices), figsize=(len(sample_indices) * 3, 3))
        if len(sample_indices) == 1: # if only one sample, axes might not be an array
            axes = [axes] 

        for i, data_idx in enumerate(sample_indices):
            image, label = current_subset_data[data_idx]
            
            unnormalized_image = image.squeeze().numpy() * 0.3081 + 0.1307 # Unnormalize
                                                                     
            ax = axes[i]
            ax.imshow(unnormalized_image, cmap='gray')
            ax.set_title(f"True: {label}")
            ax.axis('off')
            
            # Optional: Add prediction
            with torch.no_grad():
                image_tensor = image.unsqueeze(0)
                output = model(image_tensor)
                _, predicted = torch.max(output.data, 1)
                ax.set_xlabel(f"Pred: {predicted.item()}")

        plt.tight_layout()
        plt.show()
    else:
        print("No samples to display from this subset as it was empty.")
    print(f"--- End of report for Subset {subset_idx + 1} ---\n")

print('Finished Training on all subsets')
# The variable `overall_val_accuracies_after_subset_training` can be used later for plotting

In [ ]:
# Final evaluation of the model
model.eval() # Set model to evaluation mode
correct = 0
total = 0
with torch.no_grad(): # Disable gradient calculations
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

final_accuracy = 100 * correct / total
print(f'Final accuracy of the network on the {len(testset)} test images: {final_accuracy:.2f} %')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Ensure model is in evaluation mode
model.eval()

# Get a few random samples from the testset
num_samples = 5
# Create a random permutation of indices from the testset
random_indices = np.random.choice(len(testset), num_samples, replace=False)

fig, axes = plt.subplots(1, num_samples, figsize=(15, 3)) # Create a figure with subplots

for i, idx in enumerate(random_indices):
    image, label = testset[idx]
    
    # The image is a PyTorch tensor. We need to move it to CPU if it's on GPU,
    # remove the channel dimension for grayscale, and convert to numpy for plotting.
    # Also, unnormalize the image if normalization was applied during data loading.
    # The normalization for MNIST was Normalize((0.1307,), (0.3081,))
    # Unnormalization: image * std + mean
    unnormalized_image = image.squeeze().numpy() * 0.3081 + 0.1307 
                                                             
    ax = axes[i]
    ax.imshow(unnormalized_image, cmap='gray')
    ax.set_title(f"True: {label}")
    ax.axis('off') # Hide axes ticks

    # Make a prediction
    # We need to add a batch dimension (B) and channel dimension (C) to the image (H, W) -> (B, C, H, W)
    # and move it to the same device as the model if necessary.
    with torch.no_grad():
        # Assuming image is (C, H, W) and model expects (B, C, H, W)
        # Also, ensure the image tensor is on the same device as the model
        image_tensor = image.unsqueeze(0) 
        # If your model is on a GPU, move the tensor to GPU:
        # if next(model.parameters()).is_cuda:
        #    image_tensor = image_tensor.to('cuda')
            
        output = model(image_tensor)
        _, predicted = torch.max(output.data, 1)
        ax.set_xlabel(f"Pred: {predicted.item()}")

plt.tight_layout()
plt.show()

# Print predicted and true labels below the images for clarity
print("Details of the samples shown above:")
for i, idx in enumerate(random_indices):
    _, label = testset[idx] # Original label
    
    # Re-predict to match the loop structure for printing
    image_tensor = testset[idx][0].unsqueeze(0)
    # if next(model.parameters()).is_cuda:
    #    image_tensor = image_tensor.to('cuda')
        
    with torch.no_grad():
        output = model(image_tensor)
        _, predicted = torch.max(output.data, 1)
    
    print(f"Sample {i+1}: True Label = {label}, Predicted Label = {predicted.item()}")


In [ ]:
import matplotlib.pyplot as plt

# Ensure overall_val_accuracies_after_subset_training is available from the training cell
# Ensure NUM_SUBSETS is available (or derive from the length of the accuracy list)

if 'overall_val_accuracies_after_subset_training' in globals() and \
   len(overall_val_accuracies_after_subset_training) > 0:
    
    subset_numbers = range(1, len(overall_val_accuracies_after_subset_training) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(subset_numbers, overall_val_accuracies_after_subset_training, marker='o', linestyle='-')
    plt.title('Validation Accuracy on Full Test Set vs. Training Subset')
    plt.xlabel('Number of Subsets Trained On')
    plt.ylabel('Validation Accuracy (%)')
    plt.xticks(subset_numbers) # Ensure all subset numbers are ticked
    plt.grid(True)
    plt.show()
    
    print("\nSummary of Validation Accuracies per Subset:")
    for i, acc in enumerate(overall_val_accuracies_after_subset_training):
        print(f"After training on Subset {i+1}: {acc:.2f}%")
else:
    print("Training data for plotting not available. Please run the 'train-model' cell first.")